In [ ]:
from collections import defaultdict
from copy import deepcopy
import glob
import json
import os
import re

import numpy as np
import pandas as pd

from union_lists.dataset import reformat_union_lists as data
from union_lists.config import INTERIM_DATA_DIR

In [ ]:
scale = "One Inch"

### Import csv and metadata files

In [ ]:
csv_files = glob.glob(f"../data/interim/{scale}/*.csv")
metadata_files = glob.glob(f"../data/interim/{scale}/*.json")
# docx_files = [x for x in docx_files if "~" not in x and "(2)" not in x]
# docx_files = [x for x in docx_files if "_mod" not in x]

In [ ]:
scale = "One Inch"
block_suffix = {"One Inch": "A", "Half Inch": "B", "Quarter Inch": "C"}[scale]
csv_files = glob.glob(f"{INTERIM_DATA_DIR}/{scale}/*.csv")

input_dfs, metadatas = {}, {}
for f in csv_files:
    file_id = os.path.basename(f).split(".")[0] + ".doc"
    df = pd.read_csv(f, encoding="utf8").dropna(how="all")
    with open(f[:-4] + ".json") as g:
        metadata = json.load(g)
    input_dfs[file_id] = data.pre_process_df(df)
    metadatas[file_id] = metadata

data.fix_data_errors(input_dfs)

In [ ]:
input_dfs

In [ ]:
input_dfs["38A.doc"]

In [ ]:
entry_dfs = {}
for file_id, df in input_dfs.items():
    print(file_id)
    entries = []
    if df.columns.equals(pd.Index(['Post-1905_1', 'Post-1905_2', '1886-1905_1', '1886-1905_2', 'Pre-1886_1', 'Pre-1886_2'])):
        [entries.extend(data.process_6col_row(row[1], source=file_id, scale=scale, metadata=metadatas[file_id])) for row in df.iterrows()]
        entry_df = pd.concat([pd.DataFrame(x, index=[0]) for x in entries]).reset_index(drop=True)
        entry_dfs[file_id] = entry_df

In [ ]:
csv_files, metadata_files

In [ ]:
def pre_process_df(df: pd.DataFrame) -> pd.DataFrame:
    # Apply any preprocessing that can be column vectorised before the rows are processed
    for col in df.dropna(axis=1, how="all"):
        if df[col].dtype in (pd.StringDtype(na_value=np.nan), str):
            df[col] = df[col].str.replace("  ", " ")
    return df

In [ ]:
dfs, metadatas = {}, {}
for f in csv_files:
    file_id = os.path.basename(f).split(".")[0] + ".doc"
    df = pd.read_csv(f, encoding="utf8")
    with open(f[:-4] + ".json") as g:
        metadata = json.load(g)
    dfs[file_id] = pre_process_df(df)
    metadatas[file_id] = metadata

In [ ]:
ref.fix_data_errors(dfs)

In [ ]:
ref_re = re.compile(r"X/\d{1,7}/[\d\w/+]{1,}(?=\s)")

In [ ]:
combined_input_text = ""
for df in dfs.values():
    df = df.dropna(axis=1, how="all")
    df += " "
    combined_input_text += df.sum(axis=1).sum(axis=0)

In [ ]:
combined_input_text.index('X/9051/38N')

In [ ]:
combined_input_text[368000:372000]

In [ ]:
combined_input[90600:90850]

In [ ]:
y_one_df = pd.read_csv("../data/interim/One Inch/Y101 38 X9053.csv", encoding="utf-8-sig").dropna(how="all")
y_quarter_df = pd.read_csv("../data/interim/Quarter Inch/Y104 38 X9051.csv", encoding="utf-8-sig").dropna(how="all")

In [ ]:
y_one_df.drop(index=[242, 243, 244, 245, 246, 248, 249, 250, 251, 252, 254, 255, 256, 257,
       259, 260, 261, 262, 263, 264, 265, 267, 268, 269, 271, 272, 273, 274,
       275])

In [ ]:
y_one_df.iloc[-29:].index

In [ ]:
y_quarter_df.iloc[-29:].reset_index()["metadata"] == y_one_df.iloc[-29:].reset_index()["metadata"]

In [ ]:
y_one_df.iloc[-29:]

In [ ]:
entry_dfs = {}
for file_id, df in dfs.items():
    print(file_id)
    entries = []
    [entries.extend(ref.process_6col_row(row[1], source=file_id, scale="Half Inch", metadata=metadatas[file_id])) for row in df.iterrows()];
    entry_df = pd.concat([pd.DataFrame(x, index=[0]) for x in entries]).reset_index(drop=True)
    entry_dfs[file_id] = entry_df

In [ ]:
entry_dfs.keys()

In [ ]:
bn, bl, sid = "34", "B", "NE"
entry_dfs["34B.doc"].query(f"`Post-1905 Block Number` == '{bn}' and `Post-1905 Block Letter` == '{bl}' and `Post-1905 Sheet ID` == '{sid}'")["Notes"]

In [ ]:
combined_df = pd.concat([df for df in entry_dfs.values()])

In [ ]:
combined_df.info()

In [ ]:
# This sort_values arranges Half Inch quadrants in the current BL cataloguing order of NW/NE/SW/SE compared to the previous standard of NW/SW/NE/SE
# I haven't implemented it as it makes it harder to map results to the raw data by row

# .sort_values(by=["Post-1905 Block Number", "Post-1905 Block Letter", "Post-1905 Sheet ID"], key=lambda x: x.apply(lambda y:{"NW":1, "SW":3, "NE":2, "SE":4}.get(y, y)))

In [ ]:
# sample_df.to_csv("../data/processed/v0.4_sample.csv", encoding="utf-8-sig", index=False)

### Examine data model

Used to create the fields in process_6col_row

In [ ]:
data_model_df = pd.read_excel("../data/external/Data Model - Draft - for SI Union List - Populated.xlsx", sheet_name="Half Inch")
notes = data_model_df.iloc[0, :].to_dict()
data_model_df = data_model_df.drop(index=0).reset_index(drop=True)

In [ ]:
data_model_df.head(12)

In [ ]:
data_model_df.info()